# Lightweight learned provenance graph

This notebook trains the two-layer relational GraphSAGE model, not RF/XGBoost/LightGBM.

Attach the updated `learned_graph_code.zip` (or extract it into a Kaggle dataset), the completed graph index, and the approved
baseline shard package. The trainer locates shards by their manifest-relative paths even when the manifest still records
the original local absolute paths. Keep the manifest and its `assets/` directory together.

Default limits are an initial sampled experiment. For full validation/test set the
corresponding limits to zero and budget additional runtime. No result shown here is
an OpTC accuracy claim until these cells are executed and metrics inspected.


In [ ]:
from pathlib import Path
import subprocess, sys, json, os, zipfile

# --- Auto-discover dataset paths under /kaggle/input ---
# The code can be uploaded as a Kaggle dataset containing the ZIP artifact.
INPUT_ROOT = Path('/kaggle/input')
EXTRACTED_CODE = Path('/kaggle/working/provenance-code')
zip_candidates = list(INPUT_ROOT.rglob('learned_graph_code.zip'))
if zip_candidates:
    EXTRACTED_CODE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_candidates[0]) as archive:
        archive.extractall(EXTRACTED_CODE)

# 1. Discover train.py in either a directly uploaded code tree or extracted ZIP.
search_roots = [EXTRACTED_CODE, INPUT_ROOT]  # Prefer the freshly uploaded code ZIP over stale source trees
train_candidates = [p for root in search_roots if root.exists() for p in root.rglob('train.py') if 'provenace_graph' in str(p)]
if not train_candidates:
    print('Attached datasets found in /kaggle/input:')
    for item in sorted(Path('/kaggle/input').iterdir()):
        print(' -', item.name)
    raise FileNotFoundError('Could not find provenace_graph/model/train.py in /kaggle/input. Please ensure the provenance-code dataset is attached.')
train_path = train_candidates[0]
CODE_ROOT = train_path.parent.parent.parent
print(f'Discovered train.py: {train_path}')
if EXTRACTED_CODE in train_path.parents:
    print('Using the extracted learned_graph_code.zip.')
else:
    print('Using an attached source tree; it must include preprocess/build_graph.py.')
print(f'Discovered CODE_ROOT: {CODE_ROOT}')

# 2. Discover graph.sqlite and its manifest
graph_candidates = list(Path('/kaggle/input').rglob('graph.sqlite'))
if not graph_candidates:
    raise FileNotFoundError('Could not find graph.sqlite in /kaggle/input. Please ensure the provenance-graph dataset is attached.')
GRAPH_ROOT = graph_candidates[0].parent
print(f'Discovered GRAPH_ROOT: {GRAPH_ROOT}')

# 3. Discover baseline host-input manifest
manifests = [p for p in Path('/kaggle/input').rglob('manifest.json') if p.parent != GRAPH_ROOT]
baseline_candidates = []
for candidate in manifests:
    try:
        candidate_meta = json.loads(candidate.read_text())
    except (OSError, json.JSONDecodeError):
        continue
    if candidate_meta.get('format') == 'host_binary_delta_npz' and candidate_meta.get('complete') is True:
        baseline_candidates.append(candidate)
if len(baseline_candidates) != 1:
    raise RuntimeError(f'Expected exactly one complete Host shard manifest, found: {baseline_candidates}')
BASELINE_MANIFEST = baseline_candidates[0]
print(f'Discovered BASELINE_MANIFEST: {BASELINE_MANIFEST}')

OUTPUT = Path('/kaggle/working/provenance-model')
TRAIN_LIMIT = 20_000
VAL_LIMIT = 5_000   # 0 = full validation
TEST_LIMIT = 5_000  # 0 = full test
EPOCHS = 10
RESUME = False

meta = json.loads((GRAPH_ROOT / 'manifest.json').read_text())
if meta['config']['pilot']: raise ValueError('A partial pilot is not a training graph.')
print(f'Found complete Host shard manifest: {BASELINE_MANIFEST}')
print('The trainer will verify shard hashes and resolve relocated volume roots under /kaggle/input.')
print('All 3 datasets discovered and validated successfully!')


In [ ]:
import os, shutil
env = os.environ.copy()
# A failed pre-training attempt may leave only config.json. Clear only that
# empty initialization; preserve any checkpointed run for explicit --resume.
if OUTPUT.exists() and not (OUTPUT/'metrics.json').exists() and not (OUTPUT/'progress.json').exists() and not list(OUTPUT.glob('model-slot-*.npz')):
    shutil.rmtree(OUTPUT)
    print('Removed incomplete pre-training output; no model checkpoint existed.')
env['PYTHONPATH'] = f"{CODE_ROOT}:{CODE_ROOT / 'preprocess'}:{train_path.parent}:{env.get('PYTHONPATH', '')}"

command = [sys.executable, str(train_path),
           '--graph', str(GRAPH_ROOT), '--baseline', str(BASELINE_MANIFEST),
           '--output', str(OUTPUT), '--epochs', str(EPOCHS),
           '--train-limit', str(TRAIN_LIMIT), '--val-limit', str(VAL_LIMIT),
           '--test-limit', str(TEST_LIMIT), '--evaluate-test']
if RESUME: command.append('--resume')
print('Executing:', ' '.join(command))
subprocess.run(command, env=env, check=True)


In [ ]:
metrics = json.loads((OUTPUT/'metrics.json').read_text())
print(json.dumps(metrics, indent=2))
print('Inspect sampled/full scope, precision-recall and FPR before comparing to the baseline.')


Model checkpoints contain learned graph weights and Adam state. `score_event.py`
returns the two root embeddings, a risk score, actual ancestry edges and edge-ablation
sensitivity. None of these is proof of the true attack root. Preserve the seed02
split and keep test out of model selection. The risk head can be ignored when only
learned embeddings are needed by a later fusion experiment.
